# 06 — Integración ENOE × INPC → Panel Analítico (Plata)

**Propósito:** Cruzar ENOE Plata con INPC para calcular salarios reales
(deflactados, base diciembre 2018) y producir el conjunto analítico
listo para consumo en la capa Plata.

**Pipeline:** `ingesta_general.ipynb` → `05_limpieza_enoe.ipynb` → **`06_integracion.ipynb`**

**Prerequisitos:**
- `ingesta_general.ipynb` ejecutado (bronce ENOE, INPC, SHCP, IGAE)
- `05_limpieza_enoe.ipynb` ejecutado (plata ENOE limpia)

**Granularidad de salida:** persona × trimestre (misma que ENOE)
**Ubicación:** `data/silver/panel_enoe_inpc/`

In [18]:
# ── Importaciones ───────────────────────────────────────────────────────────
import os
from pathlib import Path

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, StringType

spark = (
    SparkSession.builder
    .appName('06_integracion')
    .master('local[*]')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.sql.parquet.compression.codec', 'snappy')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('Spark', spark.version)

Spark 3.5.9


In [19]:
# ── Rutas ───────────────────────────────────────────────────────────────────
BASE   = Path(os.getcwd()).parent
RAW    = BASE / 'data' / 'raw'
SILVER = BASE / 'data' / 'silver'

ENOE_SILVER  = str(SILVER / 'enoe')               # salida de 05_limpieza_enoe
INPC_PARQUET = str(RAW    / 'inpc' / 'parquet')   # bronce INPC
IGAE_PARQUET = str(RAW    / 'igae' / 'parquet')   # bronce IGAE
PANEL_OUT    = str(SILVER / 'panel_enoe_inpc')    # salida de este notebook

Path(PANEL_OUT).mkdir(parents=True, exist_ok=True)

print('ENOE Plata  :', ENOE_SILVER)
print('INPC Bronce :', INPC_PARQUET)
print('IGAE Bronce :', IGAE_PARQUET)
print('Panel salida:', PANEL_OUT)

ENOE Plata  : c:\Users\PC BEAR\Documents\GitHub\mexico-economic-pipeline\data\silver\enoe
INPC Bronce : c:\Users\PC BEAR\Documents\GitHub\mexico-economic-pipeline\data\raw\inpc\parquet
IGAE Bronce : c:\Users\PC BEAR\Documents\GitHub\mexico-economic-pipeline\data\raw\igae\parquet
Panel salida: c:\Users\PC BEAR\Documents\GitHub\mexico-economic-pipeline\data\silver\panel_enoe_inpc


---
## 1 · Extracción — leer fuentes

Se cargan ENOE (capa Plata) e INPC (capa Bronce).
Se registran conteos y granularidad de entrada.

In [20]:
# ── 1.1 Cargar ENOE Plata ───────────────────────────────────────────────────
df_enoe = spark.read.parquet(ENOE_SILVER)

filas_enoe = df_enoe.count()
print(f'ENOE Plata  — filas    : {filas_enoe:,}')
print(f'ENOE Plata  — columnas : {len(df_enoe.columns)}')
print(f'ENOE Plata  — periodo  :',
      df_enoe.agg(F.min('anio'), F.max('anio')).collect()[0][:])

ENOE Plata  — filas    : 24,540,875
ENOE Plata  — columnas : 44
ENOE Plata  — periodo  : (2005, 2026)


In [21]:
# ── 1.2 Cargar INPC (subíndice general) ────────────────────────────────────
# Solo necesitamos el INPC general para deflactar.
# El subíndice que representa el nivel general es
# 'Precios al Consumidor (INPC)' en la columna subindice.
df_inpc_raw = spark.read.parquet(INPC_PARQUET)

print('Subíndices disponibles en INPC:')
df_inpc_raw.select('subindice').distinct().orderBy('subindice').show(truncate=False)

Subíndices disponibles en INPC:
+-------------------------------------------------+
|subindice                                        |
+-------------------------------------------------+
|Agropecuarios                                    |
|Energéticos y tarifas autorizadas por el gobierno|
|Mercancías                                       |
|No subyacente                                    |
|Precios al Consumidor (INPC)                     |
|Servicios                                        |
|Subyacente                                       |
+-------------------------------------------------+



In [22]:
# ── 1.3 Filtrar INPC general y calcular promedio trimestral ─────────────────
# La ENOE tiene granularidad trimestral; el INPC es mensual.
# Se promedia el INPC general por trimestre para hacer el join.

SUBINDICE_GENERAL = 'Precios al Consumidor (INPC)'

df_inpc = (
    df_inpc_raw
    .filter(F.col('subindice') == SUBINDICE_GENERAL)
    .groupBy('anio', 'trimestre')
    .agg(F.avg('indice_valor').alias('inpc_trim'))
    .orderBy('anio', 'trimestre')
)

print(f'Trimestres INPC disponibles: {df_inpc.count()}')
df_inpc.show(20, truncate=False)

Trimestres INPC disponibles: 95
+----+---------+------------------+
|anio|trimestre|inpc_trim         |
+----+---------+------------------+
|2003|1        |53.73749784769033 |
|2003|2        |54.003605423121996|
|2003|3        |54.26902225857734 |
|2003|4        |55.12018659297167 |
|2004|1        |56.06011101417334 |
|2004|2        |56.31879313487167 |
|2004|3        |56.868449470107   |
|2004|4        |58.062911572156   |
|2005|1        |58.52657078048367 |
|2005|2        |58.858816708536665|
|2005|3        |59.12768724386834 |
|2005|4        |59.862461892447335|
|2006|1        |60.69083180766034 |
|2006|2        |60.698429947387005|
|2006|3        |61.22183816356634 |
|2006|4        |62.34359988322967 |
|2007|1        |63.17922256818233 |
|2007|2        |63.11066662564733 |
|2007|3        |63.662567865802004|
|2007|4        |64.71922734280632 |
+----+---------+------------------+
only showing top 20 rows



In [23]:
# ── 1.4 Cargar IGAE (actividad económica agregada) ──────────────────────────
# Se usará el IGAE Total promediado por trimestre para enriquecer el panel.
df_igae_raw = spark.read.parquet(IGAE_PARQUET)

df_igae = (
    df_igae_raw
    .filter(F.col('ACTIVIDAD') == 'Total')
    .withColumnRenamed('CICLO', 'anio')
    .withColumnRenamed('TRIMESTRE', 'trimestre')
    .groupBy('anio', 'trimestre')
    .agg(F.avg('IGAE').alias('igae_total'))
    .orderBy('anio', 'trimestre')
)

print(f'Trimestres IGAE disponibles: {df_igae.count()}')
df_igae.show(20, truncate=False)

Trimestres IGAE disponibles: 123
+----+---------+------------------+
|anio|trimestre|igae_total        |
+----+---------+------------------+
|1993|1        |56.93075214795076 |
|1993|2        |57.501054113942104|
|1993|3        |57.836120887864666|
|1993|4        |58.552697575764334|
|1994|1        |58.2958683069219  |
|1994|2        |60.5902497618787  |
|1994|3        |60.2016014053706  |
|1994|4        |61.602689071052794|
|1995|1        |57.92951589716896 |
|1995|2        |55.005618848764435|
|1995|3        |55.8067540460466  |
|1995|4        |57.72440549412417 |
|1996|1        |58.8493598009242  |
|1996|2        |59.47751775377554 |
|1996|3        |60.07289878418101 |
|1996|4        |62.512658719277   |
|1997|1        |61.22458411825036 |
|1997|2        |64.4818524859433  |
|1997|3        |64.97460008109756 |
|1997|4        |67.25391298679092 |
+----+---------+------------------+
only showing top 20 rows



---
## 2 · Transformación — deflactación y enriquecimiento

Se calculan:
- **`inpc_base`**: valor del INPC en el trimestre de referencia (T4-2018)
- **`salario_real`**: `ingocup × (inpc_base / inpc_trim)` — salario real base dic 2018
- **`ing7c_real`**: deflactación del ingreso de trabajo (incluye autoconsumo)

> **Decisión:** Se usa dic-2018 como base porque el IGAE usa base 2018=100,
> permitiendo comparar magnitudes entre fuentes.

In [24]:
# ── 2.1 Obtener INPC base (promedio T4-2018) ────────────────────────────────
inpc_base_row = (
    df_inpc
    .filter((F.col('anio') == 2018) & (F.col('trimestre') == 4))
    .select('inpc_trim')
    .collect()
)

if inpc_base_row:
    INPC_BASE = inpc_base_row[0]['inpc_trim']
else:
    # Fallback: usar el primer trimestre disponible como base
    INPC_BASE = df_inpc.agg(F.min('inpc_trim')).collect()[0][0]
    print('ADVERTENCIA: T4-2018 no encontrado, usando mínimo como base')

print(f'INPC base (T4-2018): {INPC_BASE:.4f}')

INPC base (T4-2018): 102.2543


In [25]:
# ── 2.2 Unir ENOE con INPC trimestral ──────────────────────────────────────
df_panel = df_enoe.join(
    df_inpc.select('anio', 'trimestre', 'inpc_trim'),
    on=['anio', 'trimestre'],
    how='left'
)

# Verificar que el join no generó filas extras (no debe haber fan-out)
filas_post_join = df_panel.count()
print(f'Filas pre-join  : {filas_enoe:,}')
print(f'Filas post-join : {filas_post_join:,}')
if filas_post_join != filas_enoe:
    print('ADVERTENCIA: el join cambió el conteo de filas')
else:
    print('OK — sin fan-out')

# Nulos en inpc_trim (trimestres ENOE sin INPC correspondiente)
nulos_inpc = df_panel.filter(F.col('inpc_trim').isNull()).count()
print(f'Trimestres sin INPC: {nulos_inpc:,} filas')

Filas pre-join  : 24,540,875
Filas post-join : 24,540,875
OK — sin fan-out
Trimestres sin INPC: 0 filas


In [26]:
# ── 2.3 Calcular salario real deflactado ────────────────────────────────────
df_panel = df_panel.withColumn(
    'salario_real',
    F.when(
        F.col('ingocup').isNotNull() & F.col('inpc_trim').isNotNull() & (F.col('inpc_trim') > 0),
        F.round(F.col('ingocup') * (F.lit(INPC_BASE) / F.col('inpc_trim')), 2)
    ).otherwise(F.lit(None).cast(DoubleType()))
).withColumn(
    'ing7c_real',
    F.when(
        F.col('ing7c').isNotNull() & F.col('inpc_trim').isNotNull() & (F.col('inpc_trim') > 0),
        F.round(F.col('ing7c') * (F.lit(INPC_BASE) / F.col('inpc_trim')), 2)
    ).otherwise(F.lit(None).cast(DoubleType()))
)

print('Columnas de salario real creadas OK')
df_panel.select('anio', 'trimestre', 'ingocup', 'inpc_trim', 'salario_real').show(10)

Columnas de salario real creadas OK
+----+---------+-------+------------------+------------+
|anio|trimestre|ingocup|         inpc_trim|salario_real|
+----+---------+-------+------------------+------------+
|2023|        1| 4300.0|127.92366666666668|     3437.16|
|2023|        4|  860.0|131.47566666666665|      668.86|
|2023|        1|12900.0|127.92366666666668|    10311.47|
|2023|        3|    0.0|           129.499|         0.0|
|2023|        1|    0.0|127.92366666666668|         0.0|
|2023|        2| 8600.0|128.22033333333334|     6858.41|
|2023|        1|    0.0|127.92366666666668|         0.0|
|2023|        3|    0.0|           129.499|         0.0|
|2023|        2|    0.0|128.22033333333334|         0.0|
|2023|        2| 5160.0|128.22033333333334|     4115.04|
+----+---------+-------+------------------+------------+
only showing top 10 rows



In [27]:
# ── 2.4 Unir IGAE (contexto macroeconómico) ────────────────────────────────
df_panel = df_panel.join(
    df_igae.select('anio', 'trimestre', 'igae_total'),
    on=['anio', 'trimestre'],
    how='left'
)

print(f'Columnas finales del panel: {len(df_panel.columns)}')
print(df_panel.columns)

Columnas finales del panel: 48
['anio', 'trimestre', 'r_def', 'cd_a', 'ent', 'con', 'upm', 'd_sem', 'n_pro_viv', 'v_sel', 'n_hog', 'h_mud', 'n_ent', 'per', 'n_ren', 'sex', 'eda', 'anios_esc', 'cs_p13_1', 'niv_ins', 'clase1', 'clase2', 'pos_ocu', 'seg_soc', 'ing7c', 'ingocup', 'hrsocup', 'rama', 'scian', 'ur', 'zona', 'salario', 'fac', 'sub_o', 'tip_con', 'p1', 'p3', 'p3b', 'p4', 'p4b', 'p4c', 'p5c_thrs', 'p5h', 'periodo', 'inpc_trim', 'salario_real', 'ing7c_real', 'igae_total']


---
## 3 · Validación del conjunto integrado

Se verifican la granularidad, integridad del join y consistencia
de las variables deflactadas antes de guardar.

In [28]:
# ── 3.1 Granularidad: una fila por persona × trimestre ──────────────────────
JOIN_KEYS = [
    'r_def', 'cd_a', 'ent', 'con', 'upm', 'd_sem',
    'n_pro_viv', 'v_sel', 'n_hog', 'h_mud', 'n_ent', 'per', 'n_ren'
]

total      = df_panel.count()
distintos  = df_panel.dropDuplicates(JOIN_KEYS + ['anio', 'trimestre']).count()
print(f'Total filas       : {total:,}')
print(f'Filas únicas      : {distintos:,}')
print(f'Duplicados        : {total - distintos:,}  (esperado: 0)')

Total filas       : 24,540,875
Filas únicas      : 24,540,875
Duplicados        : 0  (esperado: 0)


In [29]:
# ── 3.2 Verificar salario_real vs ingocup nominal ───────────────────────────
# Para ocupados (clase1=1) con ingreso > 0, el salario real debe ser positivo.
(
    df_panel
    .filter((F.col('clase1') == 1) & (F.col('ingocup') > 0))
    .select(
        F.count('*').alias('ocupados_con_ingreso'),
        F.round(F.avg('ingocup'),     2).alias('ingocup_nominal_prom'),
        F.round(F.avg('salario_real'),2).alias('salario_real_prom'),
        F.count(F.when(F.col('salario_real').isNull(), True)).alias('sin_deflactar')
    )
    .show(truncate=False)
)

+--------------------+--------------------+-----------------+-------------+
|ocupados_con_ingreso|ingocup_nominal_prom|salario_real_prom|sin_deflactar|
+--------------------+--------------------+-----------------+-------------+
|10452721            |6468.97             |7144.98          |0            |
+--------------------+--------------------+-----------------+-------------+



In [30]:
# ── 3.3 Distribución por clase de actividad ─────────────────────────────────
# clase1: 1=ocupados, 2=desocupados, 3=no económicamente activos
total_panel = df_panel.count()
(
    df_panel
    .groupBy('clase1')
    .agg(
        F.count('*').alias('filas'),
        F.round(F.count('*') * 100.0 / total_panel, 2).alias('pct')
    )
    .orderBy('clase1')
    .show()
)

+------+--------+-----+
|clase1|   filas|  pct|
+------+--------+-----+
|     0|       2|  0.0|
|     1|14864040|60.57|
|     2| 9676833|39.43|
+------+--------+-----+



In [31]:
# ── 3.4 Tendencia de salario real promedio por año (ocupados) ───────────────
print('Salario real promedio anual (ocupados con ingreso > 0):')
(
    df_panel
    .filter((F.col('clase1') == 1) & (F.col('salario_real') > 0))
    .groupBy('anio')
    .agg(
        F.round(F.avg('salario_real'), 2).alias('salario_real_prom'),
        F.round(F.avg('igae_total'),   2).alias('igae_prom'),
        F.count('*').alias('ocupados')
    )
    .orderBy('anio')
    .show(30, truncate=False)
)

Salario real promedio anual (ocupados con ingreso > 0):
+----+-----------------+---------+--------+
|anio|salario_real_prom|igae_prom|ocupados|
+----+-----------------+---------+--------+
|2005|7665.12          |77.93    |554902  |
|2006|7868.22          |81.62    |562424  |
|2007|7940.78          |83.44    |554259  |
|2008|7736.95          |83.97    |537239  |
|2009|7192.93          |78.85    |509095  |
|2010|6986.1           |82.9     |511043  |
|2011|6866.09          |85.99    |496712  |
|2012|6798.87          |89.08    |499661  |
|2013|6752.75          |89.94    |487803  |
|2014|6563.34          |92.15    |492323  |
|2015|6626.18          |94.48    |492830  |
|2016|6690.73          |96.15    |482492  |
|2017|6598.8           |98.01    |472715  |
|2018|6611.33          |100.0    |469061  |
|2019|6672.82          |98.88    |250649  |
|2020|6884.58          |95.84    |324484  |
|2021|6884.36          |96.82    |486169  |
|2022|6939.13          |100.09   |493677  |
|2023|7225.04       

---
## 4 · Almacenamiento — Panel Analítico Plata

Se guarda el panel integrado como Parquet en `data/silver/panel_enoe_inpc/`,
particionado por `anio`. Este es el conjunto de datos listo para análisis.

In [32]:
# ── 4.1 Escribir Panel Analítico Plata ──────────────────────────────────────
(
    df_panel
    .write
    .mode('overwrite')
    .partitionBy('anio')
    .parquet(PANEL_OUT)
)

size_mb = sum(
    f.stat().st_size for f in Path(PANEL_OUT).rglob('*.parquet')
) / 1_048_576
print(f'Panel Plata escrito en : {PANEL_OUT}')
print(f'Tamaño estimado        : {size_mb:.1f} MB (snappy)')

Panel Plata escrito en : c:\Users\PC BEAR\Documents\GitHub\mexico-economic-pipeline\data\silver\panel_enoe_inpc
Tamaño estimado        : 508.2 MB (snappy)


In [33]:
# ── 4.2 Validación round-trip ───────────────────────────────────────────────
df_final = spark.read.parquet(PANEL_OUT)

print('Validación round-trip:')
print(f'  Filas releídas : {df_final.count():,}')
print(f'  Columnas       : {len(df_final.columns)}')
df_final.printSchema()

Validación round-trip:
  Filas releídas : 24,540,875
  Columnas       : 48
root
 |-- trimestre: integer (nullable = true)
 |-- r_def: string (nullable = true)
 |-- cd_a: string (nullable = true)
 |-- ent: string (nullable = true)
 |-- con: string (nullable = true)
 |-- upm: string (nullable = true)
 |-- d_sem: string (nullable = true)
 |-- n_pro_viv: string (nullable = true)
 |-- v_sel: string (nullable = true)
 |-- n_hog: string (nullable = true)
 |-- h_mud: string (nullable = true)
 |-- n_ent: string (nullable = true)
 |-- per: string (nullable = true)
 |-- n_ren: string (nullable = true)
 |-- sex: integer (nullable = true)
 |-- eda: integer (nullable = true)
 |-- anios_esc: string (nullable = true)
 |-- cs_p13_1: string (nullable = true)
 |-- niv_ins: integer (nullable = true)
 |-- clase1: integer (nullable = true)
 |-- clase2: integer (nullable = true)
 |-- pos_ocu: integer (nullable = true)
 |-- seg_soc: integer (nullable = true)
 |-- ing7c: double (nullable = true)
 |-- ingocup: 

In [34]:
# ── 4.3 Resumen final del pipeline ──────────────────────────────────────────
filas_panel = df_final.count()
print('=' * 60)
print('RESUMEN — PIPELINE COMPLETO ACTIVIDAD 11')
print('=' * 60)
print()
print('ENTRADAS:')
print(f'  ENOE bronce (85 ZIPs) : {filas_enoe:,} filas')
print(f'  INPC mensual          : serie general, promediada por trimestre')
print(f'  IGAE mensual          : Total, promediado por trimestre')
print()
print('TRANSFORMACIONES:')
print('  1. ENOE: cast de tipos, dedup, filtro eda>=15, -1->NULL en ingresos')
print('  2. INPC: filtro subíndice general, promedio trimestral')
print('  3. IGAE: filtro Total, promedio trimestral')
print('  4. Join ENOE x INPC: left join por anio+trimestre')
print('  5. Join panel x IGAE: left join por anio+trimestre')
print('  6. Deflactación: salario_real e ing7c_real (base T4-2018)')
print()
print('SALIDA:')
print(f'  Panel analítico       : {filas_panel:,} filas')
print(f'  Columnas              : {len(df_final.columns)}')
rango = df_final.select(F.min('anio'), F.max('anio')).collect()[0]
print(f'  Cobertura temporal    : {rango[0]} – {rango[1]}')
print(f'  Granularidad          : persona x trimestre')
print(f'  Ubicación             : data/silver/panel_enoe_inpc/')
print('=' * 60)
print('Pipeline completado — datos listos para análisis.')
spark.stop()

RESUMEN — PIPELINE COMPLETO ACTIVIDAD 11

ENTRADAS:
  ENOE bronce (85 ZIPs) : 24,540,875 filas
  INPC mensual          : serie general, promediada por trimestre
  IGAE mensual          : Total, promediado por trimestre

TRANSFORMACIONES:
  1. ENOE: cast de tipos, dedup, filtro eda>=15, -1->NULL en ingresos
  2. INPC: filtro subíndice general, promedio trimestral
  3. IGAE: filtro Total, promedio trimestral
  4. Join ENOE x INPC: left join por anio+trimestre
  5. Join panel x IGAE: left join por anio+trimestre
  6. Deflactación: salario_real e ing7c_real (base T4-2018)

SALIDA:
  Panel analítico       : 24,540,875 filas
  Columnas              : 48
  Cobertura temporal    : 2005 – 2026
  Granularidad          : persona x trimestre
  Ubicación             : data/silver/panel_enoe_inpc/
Pipeline completado — datos listos para análisis.
